# 03 - Pré-processamento e Transformação

**Objetivo:** Preparar os dados sem vazamento de informação.


In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

DATA_DIR = '../'
TRAIN_FILE = DATA_DIR + 'abalone_dataset.csv'
OUTPUT_DIR = './pipeline_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(TRAIN_FILE)
print('Dataset carregado:', df.shape)


Dataset carregado: (3132, 9)


## 1. Definição de X e y


In [2]:
X = df.drop('type', axis=1)
y = df['type']

print('Features:', X.columns.tolist())
print('Distribuição do target:')
print(y.value_counts().sort_index())


Features: ['sex', 'length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight']
Distribuição do target:
type
1    1078
2    1003
3    1051
Name: count, dtype: int64


## 2. Split Estratificado


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print('Treino:', X_train.shape)
print('Validação:', X_val.shape)
print('\nClasses no treino:')
print((y_train.value_counts(normalize=True) * 100).round(2))
print('\nClasses na validação:')
print((y_val.value_counts(normalize=True) * 100).round(2))


Treino: (2505, 8)
Validação: (627, 8)

Classes no treino:
type
1    34.41
3    33.57
2    32.02
Name: proportion, dtype: float64

Classes na validação:
type
1    34.45
3    33.49
2    32.06
Name: proportion, dtype: float64


## 3. Definição das Colunas


In [4]:
cat_cols = ['sex']
num_cols = ['length', 'diameter', 'height', 'whole_weight',
        'shucked_weight', 'viscera_weight', 'shell_weight']

print('Categóricas:', cat_cols)
print('Numéricas:', num_cols)


Categóricas: ['sex']
Numéricas: ['length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight']


## 4. Pipeline de Pré-processamento

Baseline conservador: One-Hot em `sex` + StandardScaler nas numéricas.
A EDA sugeriu forte correlação entre features, então não vamos remover nada ainda; primeiro vamos validar o baseline.


In [5]:
def make_one_hot():
    try:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

def ohe_names(ohe, cols):
    if hasattr(ohe, 'get_feature_names_out'):
        return ohe.get_feature_names_out(cols).tolist()
    return ohe.get_feature_names(cols).tolist()

# Baseline de pré-processamento
# Mantém tudo para não perder sinal, já que EDA mostrou forte estrutura ordinal
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', make_one_hot(), cat_cols),
        ('num', StandardScaler(), num_cols),
    ]
)

print(preprocessor)


ColumnTransformer(transformers=[('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['sex']),
                                ('num', StandardScaler(),
                                 ['length', 'diameter', 'height',
                                  'whole_weight', 'shucked_weight',
                                  'viscera_weight', 'shell_weight'])])


## 5. Ajuste e Transformação


In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

cat_feature_names = ohe_names(preprocessor.named_transformers_['cat'], cat_cols)
processed_feature_names = cat_feature_names + num_cols

print('Train shape:', X_train_processed.shape)
print('Val shape:', X_val_processed.shape)
print('\nFeatures geradas:', processed_feature_names)
print('\nPrimeiras linhas processadas:')
print(pd.DataFrame(X_train_processed[:5], columns=processed_feature_names).round(3))


Train shape: (2505, 9)
Val shape: (627, 9)

Features geradas: ['sex_I', 'sex_M', 'length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight']

Primeiras linhas processadas:
   sex_I  sex_M  length  diameter  height  whole_weight  shucked_weight  \
0    0.0    0.0   1.599     1.590   1.051         1.955           2.173   
1    0.0    0.0   1.558     1.339   1.687         1.531           1.985   
2    0.0    0.0  -0.600    -0.518  -0.857        -0.611          -0.539   
3    1.0    0.0  -0.600    -0.568  -0.730        -0.628          -0.467   
4    0.0    1.0   0.811     0.837   0.797         0.782           0.719   

   viscera_weight  shell_weight  
0           1.577         2.148  
1           0.975         1.410  
2          -0.567        -0.657  
3          -0.676        -0.693  
4           0.690         0.697  


## 6. Verificação de Vazamento


In [7]:
cat_size = len(cat_feature_names)
num_train = X_train_processed[:, cat_size:]
num_val = X_val_processed[:, cat_size:]

print('Média treino pós-scaling:', num_train.mean(axis=0).round(3))
print('Desvio treino pós-scaling:', num_train.std(axis=0).round(3))
print('\nMédia validação:', num_val.mean(axis=0).round(3))
print('Desvio validação:', num_val.std(axis=0).round(3))


Média treino pós-scaling: [-0. -0. -0.  0.  0.  0. -0.]
Desvio treino pós-scaling: [1. 1. 1. 1. 1. 1. 1.]

Média validação: [-0.038 -0.036 -0.054 -0.057 -0.05  -0.038 -0.057]
Desvio validação: [1.009 0.998 0.986 0.953 0.944 0.952 0.96 ]


## 7. Salvamento dos Artefatos

Além do pré-processador, vamos deixar pronto o baseline para comparação posterior.


In [8]:
joblib.dump(preprocessor, OUTPUT_DIR + 'preprocessor.pkl')
np.save(OUTPUT_DIR + 'X_train_processed.npy', X_train_processed)
np.save(OUTPUT_DIR + 'X_val_processed.npy', X_val_processed)
np.save(OUTPUT_DIR + 'y_train.npy', y_train.to_numpy())
np.save(OUTPUT_DIR + 'y_val.npy', y_val.to_numpy())

print('Arquivos salvos em:', OUTPUT_DIR)
print(sorted(os.listdir(OUTPUT_DIR)))


Arquivos salvos em: ./pipeline_output/
['X_train_processed.npy', 'X_val_processed.npy', 'preprocessor.pkl', 'y_train.npy', 'y_val.npy']


## 8. Resumo

- `sex` → One-Hot Encoding
- Numéricas → StandardScaler
- Sem remoção de features por enquanto
- Split estratificado 80/20
- Artefatos salvos em `pipeline_output/`


In [9]:
print('=' * 60)
print('RESUMO DO PRÉ-PROCESSAMENTO')
print('=' * 60)
print('Pré-processamento pronto para modelagem.')
print('Próximo passo: Notebook 04 - Modelagem e validação')
print('=' * 60)


RESUMO DO PRÉ-PROCESSAMENTO
Pré-processamento pronto para modelagem.
Próximo passo: Notebook 04 - Modelagem e validação
